# 👟 ARShoe M1v2 13-Keypoint Training on `shuffled_v4_1` in Google Colab

This notebook trains **ARShoe M1v2 (13-KP)** on the new **`shuffled_v4_1`** dataset with:
1. **Automatic 13-KP Conversion**: Converts `/content/shoes_vto_v2/shoes_vto/data/shuffled_v4_1` to 13-KP with fast symlinked images.
2. **Spatial Visibility Masking**: Heatmap loss with spatial ignore masks for unannotated regions on multi-foot images.
3. **Live Test Visualizations Every 10 Epochs**: Side-by-side comparison panels rendered directly in the notebook output during training.
4. **Canonical Validation Evaluation**: Full validation split evaluation with Hungarian class-independent matching.

### Step 1: Verify GPU Environment & Install Dependencies

In [ ]:
!nvidia-smi
!pip install -q pyyaml tqdm albumentations opencv-python matplotlib scipy

### Step 2: Set Working Directory
Ensure we are located in `/content/shoes_vto_v2/shoes_vto`.

In [ ]:
import os
if os.path.exists('/content/shoes_vto_v2/shoes_vto'):
    %cd /content/shoes_vto_v2/shoes_vto
elif os.path.exists('/content/shoes_vto'):
    %cd /content/shoes_vto
print('Working Directory:', os.getcwd())

### Step 3: Convert `shuffled_v4_1` (16-KP) to 13-KP Format
- Strips `ball_top` (5), `ankle_center` (12), and `shin_mid` (15)
- Uses instant symlinks for images to avoid duplicating storage
- Output path: `/content/shoes_vto_v2/shoes_vto/data/shuffled_v4_1_13kp`

In [ ]:
!python shoes_vto/tools/convert_dataset_to_13kp.py \
    --src /content/shoes_vto_v2/shoes_vto/data/shuffled_v4_1 \
    --dst /content/shoes_vto_v2/shoes_vto/data/shuffled_v4_1_13kp \
    --symlink

### Step 4: Launch 100-Epoch Training with Live Test Visualizations Every 10 Epochs
- **Spatial visibility masking** enabled for multi-instance keypoint supervision.
- **Live Visualizations**: Every 10 epochs (`--vis_interval 10`), a 4-sample side-by-side comparison (`[Ground Truth | Model Prediction]`) will be saved and rendered right below this cell!

In [ ]:
!python shoes_vto/train_m1v2_13kp.py \
    --dataset_root /content/shoes_vto_v2/shoes_vto/data/shuffled_v4_1_13kp \
    --output_dir shoes_vto/outputs/m1v2_13kp_shuffled_v4_1_training \
    --epochs 100 \
    --batch_size 16 \
    --lr 0.0005 \
    --vis_interval 10

### Step 5: Canonical Validation Split Evaluation
Evaluates the new `best.pth` checkpoint on the validation split with Hungarian matching.

In [ ]:
!python shoes_vto/tools/evaluate_decoders_on_validation.py \
    --checkpoint shoes_vto/outputs/m1v2_13kp_shuffled_v4_1_training/checkpoints/best.pth \
    --valid_dir /content/shoes_vto_v2/shoes_vto/data/shuffled_v4_1_13kp/valid \
    --out_dir shoes_vto/outputs/validation_eval_shuffled_v4_1

### Step 6: Test Images Inspection (4 Test Images)
Generates detailed 3-panel comparisons and composite poster for the 4 test images.

In [ ]:
!python shoes_vto/tools/eval_and_vis_4_test_images_13kp.py \
    --checkpoint shoes_vto/outputs/m1v2_13kp_shuffled_v4_1_training/checkpoints/best.pth \
    --out_dir shoes_vto/outputs/4images_shuffled_v4_1_eval

### Step 7: Package Checkpoints & Visualizations for Download

In [ ]:
!zip -r m1v2_13kp_shuffled_v4_1_results.zip \
    shoes_vto/outputs/m1v2_13kp_shuffled_v4_1_training/checkpoints/best.pth \
    shoes_vto/outputs/m1v2_13kp_shuffled_v4_1_training/epoch_visualizations/ \
    shoes_vto/outputs/validation_eval_shuffled_v4_1/ \
    shoes_vto/outputs/4images_shuffled_v4_1_eval/
print('✅ Results packaged to m1v2_13kp_shuffled_v4_1_results.zip!')